# Part A: Device Query


In [10]:
%%writefile device_query.cu
// Part A: CUDA Device Query
#include <cuda_runtime.h>
#include <stdio.h>

#include <stdio.h>
#include <cuda_runtime.h>

int main() {
    int deviceCount;
    cudaGetDeviceCount(&deviceCount);

    printf("Number of CUDA devices: %d\n\n", deviceCount);

    for (int i = 0; i < deviceCount; i++) {
        cudaDeviceProp prop;
        cudaGetDeviceProperties(&prop, i);

        printf("===== Device %d: %s =====\n", i, prop.name);

        printf("Compute Capability: %d.%d\n", prop.major, prop.minor);

        printf("\n[Block & Grid]\n");
        printf("Max Threads per Block: %d\n", prop.maxThreadsPerBlock);
        printf("Max Block Dimensions: %d x %d x %d\n",
               prop.maxThreadsDim[0], prop.maxThreadsDim[1], prop.maxThreadsDim[2]);
        printf("Max Grid Dimensions: %d x %d x %d\n",
               prop.maxGridSize[0], prop.maxGridSize[1], prop.maxGridSize[2]);

        printf("\n[Memory]\n");
        printf("Global Memory: %.2f GB\n", prop.totalGlobalMem / (1024.0 * 1024 * 1024));
        printf("Shared Memory per Block: %ld KB\n", prop.sharedMemPerBlock / 1024);
        printf("Constant Memory: %ld KB\n", prop.totalConstMem / 1024);

        printf("\n[Warp]\n");
        printf("Warp Size: %d\n", prop.warpSize);

        printf("\n[Multiprocessor]\n");
        printf("Multiprocessors: %d\n", prop.multiProcessorCount);
        printf("Max Threads per Multiprocessor: %d\n", prop.maxThreadsPerMultiProcessor);

        printf("\n[Double Precision]\n");
        printf("Supported: %s\n", (prop.major >= 1) ? "Yes" : "No");

        printf("\n-----------------------------------\n\n");
    }

    return 0;
}


Overwriting device_query.cu


In [11]:
!nvcc device_query.cu -o device_query && ./device_query

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Number of CUDA devices: 1

===== Device 0: Tesla T4 =====
Compute Capability: 7.5

[Block & Grid]
Max Threads per Block: 1024
Max Block Dimensions: 1024 x 1024 x 64
Max Grid Dimensions: 2147483647 x 65535 x 65535

[Memory]
Global Memory: 14.56 GB
Shared Memory per Block: 48 KB
Constant Memory: 64 KB

[Warp]
Warp Size: 32

[Multiprocessor]
Multiprocessors: 40
Max Threads per Multiprocessor: 1024

[Double Precision]
Supported: Yes

-----------------------------------



## Part B: Array Sum

In [12]:
%%writefile array_sum.cu
#include <stdio.h>
#include <cuda.h>
#include <stdlib.h>
#include <time.h>

#define N (1 << 20)
#define BLOCK_SIZE 256

__global__ void sumKernel(float *input, float *output) {
    __shared__ float shared[BLOCK_SIZE];

    int tid = threadIdx.x;
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    shared[tid] = input[i];
    __syncthreads();

    for (int s = blockDim.x / 2; s > 0; s >>= 1) {
        if (tid < s) {
            shared[tid] += shared[tid + s];
        }
        __syncthreads();
    }

    if (tid == 0) {
        output[blockIdx.x] = shared[0];
    }
}

float cpuSum(float *arr) {
    float sum = 0.0f;
    for (int i = 0; i < N; i++) {
        sum += arr[i];
    }
    return sum;
}

int main() {
    float *h_input = (float*)malloc(N * sizeof(float));
    float *h_output;

    for (int i = 0; i < N; i++) {
        h_input[i] = 1.0f;
    }

    int gridSize = (N + BLOCK_SIZE - 1) / BLOCK_SIZE;
    h_output = (float*)malloc(gridSize * sizeof(float));

    float *d_input, *d_output;
    cudaMalloc(&d_input, N * sizeof(float));
    cudaMalloc(&d_output, gridSize * sizeof(float));

    cudaMemcpy(d_input, h_input, N * sizeof(float), cudaMemcpyHostToDevice);

    // CPU Timing
    clock_t start_cpu = clock();
    float cpu_result = cpuSum(h_input);
    clock_t end_cpu = clock();

    float cpu_time = (float)(end_cpu - start_cpu) / CLOCKS_PER_SEC * 1000;

    // GPU Timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    sumKernel<<<gridSize, BLOCK_SIZE>>>(d_input, d_output);

    cudaMemcpy(h_output, d_output, gridSize * sizeof(float), cudaMemcpyDeviceToHost);

    float gpu_result = 0.0f;
    for (int i = 0; i < gridSize; i++) {
        gpu_result += h_output[i];
    }

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, start, stop);

    printf("===== Array Sum =====\n");
    printf("Array Size: %d\n", N);
    printf("CPU Result: %f\n", cpu_result);
    printf("GPU Result: %f\n", gpu_result);

    printf("\nCPU Time: %.4f ms\n", cpu_time);
    printf("GPU Time: %.4f ms\n", gpu_time);
    printf("Speedup: %.2fx\n", cpu_time / gpu_time);

    cudaFree(d_input);
    cudaFree(d_output);
    free(h_input);
    free(h_output);

    return 0;
}


Overwriting array_sum.cu


In [13]:
!nvcc array_sum.cu -o array_sum && ./array_sum

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
===== Array Sum =====
Array Size: 1048576
CPU Result: 1048576.000000
GPU Result: 1048576.000000

CPU Time: 3.0350 ms
GPU Time: 16.4526 ms
Speedup: 0.18x


## Part C: Matrix Addition

In [14]:
%%writefile matrix_add.cu
#include <stdio.h>
#include <cuda.h>
#include <stdlib.h>
#include <time.h>

#define N 1024

__global__ void matrixAdd(int *A, int *B, int *C, int width) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < width && col < width) {
        int idx = row * width + col;
        C[idx] = A[idx] + B[idx];
    }
}

void cpuMatrixAdd(int *A, int *B, int *C) {
    for (int i = 0; i < N * N; i++) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    int size = N * N * sizeof(int);

    int *h_A = (int*)malloc(size);
    int *h_B = (int*)malloc(size);
    int *h_C = (int*)malloc(size);
    int *h_C_cpu = (int*)malloc(size);

    for (int i = 0; i < N * N; i++) {
        h_A[i] = 1;
        h_B[i] = 2;
    }

    int *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 block(32, 32);
    dim3 grid((N + block.x - 1) / block.x, (N + block.y - 1) / block.y);

    // CPU Timing
    clock_t start_cpu = clock();
    cpuMatrixAdd(h_A, h_B, h_C_cpu);
    clock_t end_cpu = clock();

    float cpu_time = (float)(end_cpu - start_cpu) / CLOCKS_PER_SEC * 1000;

    // GPU Timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    matrixAdd<<<grid, block>>>(d_A, d_B, d_C, N);

    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, start, stop);

    // Verification
    int errors = 0;
    for (int i = 0; i < N * N; i++) {
        if (h_C[i] != h_C_cpu[i]) {
            errors++;
        }
    }

    printf("===== Matrix Addition =====\n");
    printf("Matrix Size: %d x %d\n", N, N);

    int total = N * N;

    printf("\nFLOPs: %d\n", total);
    printf("Global Reads: %d\n", 2 * total);
    printf("Global Writes: %d\n", total);

    printf("\nCPU Time: %.4f ms\n", cpu_time);
    printf("GPU Time: %.4f ms\n", gpu_time);
    printf("Speedup: %.2fx\n", cpu_time / gpu_time);

    printf("\nVerification: %s (errors: %d)\n",
           (errors == 0) ? "PASS" : "FAIL", errors);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C);
    free(h_C_cpu);

    return 0;
}


Overwriting matrix_add.cu


In [15]:
!nvcc matrix_add.cu -o matrix_add && ./matrix_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
===== Matrix Addition =====
Matrix Size: 1024 x 1024

FLOPs: 1048576
Global Reads: 2097152
Global Writes: 1048576

CPU Time: 4.7190 ms
GPU Time: 14.8788 ms
Speedup: 0.32x

Verification: PASS (errors: 0)
